In [ ]:
# Cell 1 - Imports 
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from skimage.segmentation import slic
from sklearn.neighbors import kneighbors_graph
import scipy.io as sio
import os

Saving Salinas_corrected.mat to Salinas_corrected.mat
Saving Salinas_gt.mat to Salinas_gt.mat
Dataset loaded successfully!
HSI shape: (512, 217, 204)
GT shape: (512, 217)
Epoch 1, Loss: 17891.8703
Epoch 2, Loss: 4893.3963
Epoch 3, Loss: 2784.8543
Epoch 4, Loss: 2241.9542
Epoch 5, Loss: 1822.8669
Epoch 6, Loss: 1559.5526
Epoch 7, Loss: 1401.7309
Epoch 8, Loss: 1310.9200
Epoch 9, Loss: 1018.8600
Epoch 10, Loss: 1078.1990


In [ ]:

def load_dataset_auto():
    from google.colab import files
    uploaded = files.upload()

    file_list = list(uploaded.keys())

    hsi, gt = None, None

    for file in file_list:
        if file.endswith('.mat'):
            data = sio.loadmat(file)
            keys = list(data.keys())

            # Detect Salinas
            if 'salinas_corrected' in keys:
                hsi = data['salinas_corrected']
            if 'salinas_gt' in keys:
                gt = data['salinas_gt']

        elif file.endswith('.npy'):
            arr = np.load(file)
            if 'gt' in file.lower():
                gt = arr
            else:
                hsi = arr

    if hsi is None or gt is None:
        raise ValueError("Could not detect dataset properly. Please upload correct files.")

    print("Dataset loaded successfully!")
    print("HSI shape:", hsi.shape)
    print("GT shape:", gt.shape)

    return hsi, gt

In [ ]:
class PCAReducer:
    def __init__(self, n_components=30):
        self.pca = PCA(n_components=n_components)

    def fit_transform(self, X):
        H, W, C = X.shape
        X_reshaped = X.reshape(-1, C)
        X_pca = self.pca.fit_transform(X_reshaped)
        return X_pca.reshape(H, W, -1)


In [ ]:
def extract_patches(X, y, patch_size=7):
    pad = patch_size // 2
    X_pad = np.pad(X, ((pad,pad),(pad,pad),(0,0)), mode='reflect')
    patches, labels = [], []

    H, W = y.shape
    for i in range(H):
        for j in range(W):
            if y[i, j] == 0:
                continue
            patch = X_pad[i:i+patch_size, j:j+patch_size]
            patches.append(patch)
            labels.append(y[i, j]-1)

    return np.array(patches), np.array(labels)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, 3, padding=1)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        return x.flatten(2).transpose(1,2)

In [ ]:
class ViTEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.layer = nn.TransformerEncoderLayer(d_model=dim, nhead=4, batch_first=True)

    def forward(self, x):
        return self.layer(x).mean(dim=1)

In [ ]:
def build_graph(X, n_segments=200):
    segments = slic(X, n_segments=n_segments, compactness=10)
    features = []
    for seg_id in np.unique(segments):
        features.append(X[segments == seg_id].mean(axis=0))
    features = np.array(features)
    adj = kneighbors_graph(features, 5).toarray()
    return torch.tensor(features).float(), torch.tensor(adj).float()

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A):
        A = A + torch.eye(A.size(0)).to(A.device)
        D = torch.diag(torch.sum(A, dim=1))
        D_inv_sqrt = torch.inverse(torch.sqrt(D))
        A_hat = D_inv_sqrt @ A @ D_inv_sqrt
        return F.relu(self.linear(A_hat @ X))


In [ ]:
class GraphTransformer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, 4, batch_first=True)
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        x,_ = self.attn(x, x, x)
        return self.fc(x.mean(dim=1))

In [ ]:
class HSIModel(nn.Module):
    def __init__(self, in_channels, dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, dim)
        self.vit = ViTEncoder(dim)
        self.gcn = GCNLayer(in_channels, dim)
        self.graph_trans = GraphTransformer(dim)
        self.fusion = nn.Linear(dim*2, dim)
        self.cls = nn.Linear(dim, num_classes)

    def forward(self, patch, graph_nodes, adj):
        vit_feat = self.vit(self.patch_embed(patch))
        gcn_feat = self.gcn(graph_nodes, adj)
        graph_feat = self.graph_trans(gcn_feat.unsqueeze(0))
        fused = self.fusion(torch.cat([vit_feat, graph_feat], dim=-1))
        return self.cls(fused)

In [ ]:
def train_and_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # AUTO LOAD
    hsi, gt = load_dataset_auto()

    num_classes = int(np.max(gt))

    pca = PCAReducer(30)
    X_pca = pca.fit_transform(hsi)

    X, y = extract_patches(X_pca, gt, 7)
    X = torch.tensor(X).permute(0,3,1,2).float()
    y = torch.tensor(y).long()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

    class_counts = np.bincount(y_train.numpy(), minlength=num_classes)
    weights = 1.0 / (class_counts + 1e-6)
    weights = weights / weights.sum()
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    graph_nodes, adj = build_graph(X_pca)
    graph_nodes, adj = graph_nodes.to(device), adj.to(device)

    model = HSIModel(30, 64, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    for epoch in range(15):
        model.train()
        total_loss = 0
        for i in range(len(X_train)):
            patch = X_train[i].unsqueeze(0).to(device)
            label = y_train[i].unsqueeze(0).to(device)

            optimizer.zero_grad()
            out = model(patch, graph_nodes, adj)
            loss = loss_fn(out, label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(len(X_test)):
            patch = X_test[i].unsqueeze(0).to(device)
            out = model(patch, graph_nodes, adj)
            preds.append(out.argmax(dim=1).item())

    y_true = y_test.numpy()
    y_pred = np.array(preds)

    print("\n===== Metrics =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    import matplotlib.pyplot as plt

    report = classification_report(y_true, y_pred, output_dict=True)

    classes = []
    precision_vals = []
    recall_vals = []
    f1_vals = []

    for cls in report.keys():
        if cls.isdigit():  # ignore avg/total rows
            classes.append(int(cls))
            precision_vals.append(report[cls]['precision'])
            recall_vals.append(report[cls]['recall'])
            f1_vals.append(report[cls]['f1-score'])

    x = np.arange(len(classes))

    plt.figure(figsize=(12,6))
    plt.bar(x - 0.2, precision_vals, width=0.2, label='Precision')
    plt.bar(x, recall_vals, width=0.2, label='Recall')
    plt.bar(x + 0.2, f1_vals, width=0.2, label='F1-Score')

    plt.xlabel("Class Index")
    plt.ylabel("Score")
    plt.title("Per-Class Performance (Precision, Recall, F1-score)")
    plt.xticks(x, classes)
    plt.legend()
    plt.grid()

    plt.show()

In [ ]:
train_and_test()